# OpenPlaque — LAD Proximal Recenter Continuation

Continue only from the previously validated LAD proximal endpoint. Every new step is recentered on a coronary-sized component in a true source-resolution orthogonal plane. TotalSegmentator is used only for the aorta exclusion/distance constraint. Research use only.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Reuse controls — True reuses a valid cache; False forces recomputation and overwrite.
REUSE_SOURCE_CT = True
REUSE_VALIDATED_PRIOR = True
REUSE_AORTA_CONSTRAINT = True
REUSE_CONTINUATION = True
REUSE_FIGURES = True
REUSE_REPORT = True


## Step 3 — Install dependencies


In [ ]:
%pip -q install pydicom SimpleITK scipy matplotlib pandas psutil 'pylibjpeg>=2.0' 'pylibjpeg-libjpeg>=2.1'
print('JPEG Lossless decoder installed for pydicom.')


## Step 4 — Load this branch


In [ ]:
import os, sys, subprocess, shutil
REPO='/content/OpenPlaque'
BRANCH='lad-proximal-recenter-backtrack-from-main'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','--depth','1','--branch',BRANCH,'https://github.com/pazzani/OpenPlaque.git',REPO],check=True)
sys.path.insert(0, os.path.join(REPO,'src'))
print('Loaded', BRANCH)


## Step 5 — Initialize workflow


In [ ]:
from openplaque.lad_proximal_recenter import LADProximalRecenterWorkflow
reuse = {
    'source_ct': REUSE_SOURCE_CT,
    'validated_prior': REUSE_VALIDATED_PRIOR,
    'aorta_constraint': REUSE_AORTA_CONSTRAINT,
    'continuation': REUSE_CONTINUATION,
    'figures': REUSE_FIGURES,
    'report': REUSE_REPORT,
}
wf = LADProximalRecenterWorkflow(reuse=reuse)
display(wf.cache_status())


## Step 6 — Load/reuse source CCTA


In [ ]:
ct = wf.load_source_ct()
print('source CT shape:', ct.shape, 'spacing zyx mm:', wf.spacing)


## Step 7 — Revalidate the established LAD with the validated RCA calibration


In [ ]:
established = wf.validate_established_lad()
print(established)
if not established.get('accepted', False):
    raise RuntimeError('Established LAD failed the corrected source-resolution validation gate; continuation will not run.')


## Step 8 — Load TotalSegmentator aorta constraint


In [ ]:
wf.load_aorta_constraint()
print('Aorta constraint loaded. It is used only for exclusion/distance, not coronary segmentation.')


## Step 9 — Continue proximally with broadened directions + lumen recentering


In [ ]:
summary = wf.continue_proximally(max_extension_mm=42.0, beam_width=9)
print(summary)
print('Current proximal endpoint:', wf.endpoint)


## Step 10 — Generate QC figures


In [ ]:
figs = wf.plot_qc()
for f in figs: print(f)


## Step 11 — Package report-back ZIP


In [ ]:
z = wf.package()
print('REPORT_BACK:', z)
print('Expected filename: OPENPLAQUE_LAD_RECENTER_CONTINUATION_REPORT_BACK.zip')


## Interpretation gate

Do not call the proximal endpoint the LAD origin merely because tracking continued. Accept the new extension only if serial source-resolution cross-sections remain coronary-sized and centered. If the run reaches the aortic-root constraint, inspect for a left-main transition/bifurcation before assigning an LAD origin.
